# Add new parameters and constraints in ES

In [1]:
import pandas as pd
from shared.utils import load_snapshot
from energyscope.models import Model

YEAR: year of optimization
TEC: technology
MAT: material

Parameters:
- `material_intensity[YEAR, TEC, MAT]` [kt / GW]
- `limit_material_year[YEAR, MAT]` [kt]
- `limit_material[MAT]` [kt]
- `recycling_rate[YEAR, (TEC), MAT]` [-]

Variables:
- `Material_content_year[YEAR, TEC, MAT]` [kt]
- `Material_content[TEC, MAT]` [kt]
- `Recycled_material[YEAR, TEC, MAT]` [kt]

Constraints:
- `Material_content_year[YEAR, TEC, MAT] = material_intensity[YEAR, TEC, MAT] * F_new[YEAR, TEC, MAT]`
- ...

In [3]:
df = pd.read_excel("excel_files/technologies_mi_all_years.xlsx")

In [4]:
def create_dat_file_from_excel(df, file_name):
    out_path = f'ampl_files/{file_name}.dat'
    # use utf-8-sig so Windows Notepad shows accents correctly; use 'utf-8' if BOM is not desired
    with open(out_path, 'w', encoding='utf-8', newline='\n') as f:
        f.write("data;\n\n")
        f.write("set MATERIALS := Al B Cd Cr Co Concrete Cu Dy Ga Glass Ge Hf In Fe Pb Polymers Li Mg Mn Mo Nd Ni Nb Pd Pr Pt Se Si Ag Ta Te Tb Sn W V Y Zn Zr ;\n \n")
        for _, row in df.iterrows():
            value = row['Value']
            if pd.isna(value):
                continue  # skip missing values, params already default to 0
            param_name = row['Parameter']
            index0 = row['index0']
            index1 = row['index1']
            index2 = row['index2']
            unit = '-' if pd.isna(row.get('Unit')) else str(row.get('Unit'))
            comment = '' if pd.isna(row.get('Comment')) else str(row.get('Comment'))
            if pd.isna(index1) and pd.isna(index2):
                f.write(f"let {param_name}['{index0}'] := {value} ; # [{unit}] {comment}\n")
            elif pd.isna(index2):
                f.write(f"let {param_name}['{index0}','{index1}'] := {value} ; # [{unit}] {comment}\n")
            else:
                f.write(f"let {param_name}['{index0}','{index1}','{index2}'] := {value} ; # [{unit}] {comment}\n")


In [5]:
create_dat_file_from_excel(df, 'Material_intensity')

In [1]:
# Add your files to the main model
# main_model = load_snapshot(2050) + Model([('mod', 'ampl_files/test.mod'),('dat', 'ampl_files/test.dat'),])

In [2]:
from run_build_mi import main
main()                        # équivalent aux valeurs par défaut du CLI
#main(scenario='optimiste')    # applique les overrides du scénario "optimiste"
#main(write_dat=False)         # ne régénère que le xlsx

[build_table] skipping (not yet in QC_data.dat): ['NEW_WIND_OFFSHORE']
[mapping] Note: ['NEW_WIND_OFFSHORE'] are in tech_mapping.xlsx but not yet in QC_data.dat -- pre-filled for later, skipped when writing output.
[build_table] computed 36 tech intensities in 1.5s
[build_table] kept 174496 existing non-electricity rows in 48.1s
[build_table] built 9310 electricity rows in 48.3s
[build_table] wrote technologies_mi_all_years.xlsx (183806 rows) in 105.4s
[build_table] wrote Material_intensity.dat in 123.5s
[build_table] total: 123.5s
[mapping] Note: ['NEW_WIND_OFFSHORE'] are in tech_mapping.xlsx but not yet in QC_data.dat -- pre-filled for later, skipped when writing output.

Coverage report:
  not_mapped       : 3
  not_yet_modeled  : 1
  placeholder_zero : 0
  integrated       : 32

not_mapped (3):
  - PAFC
  - PEMFC
  - SOFC

not_yet_modeled (1):
  - NEW_WIND_OFFSHORE

integrated (32):
  - AFC
  - CCGT
  - CCGT_BIOGAS
  - CCGT_BIOGAS_CC
  - CCGT_CC
  - COAL_IGCC
  - COAL_IGCC_CC
  - C